# Bronze data

In [3]:
import pandas as pd
import sqlite3
import os
import time

# 1. Configuration
BASE_PATH = r'C:\Users\efloresmendoza\OneDrive - John Holland Group\2026\sql data with bara\sql-data-warehouse-project\datasets'
DB_NAME = 'Datawarehouse.db'

# Map Folder -> Files
data_map = {
    'source_crm': ['cust_info', 'prd_info', 'sales_details'],
    'source_erp': ['loc_a101', 'cust_az12', 'px_cat_g1v2']
}

# 2. Connect to (or create) the SQLite Database
conn = sqlite3.connect(DB_NAME)

print("--- Starting Load to SQLite ---")
total_start = time.time()

# 3. Loop through folders and load files
for folder, files in data_map.items():
    for file_name in files:
        file_path = os.path.join(BASE_PATH, folder, f"{file_name}.csv")

        if os.path.exists(file_path):
            start_time = time.time()

            # Read CSV - Pandas automatically guesses data types!
            df = pd.read_csv(file_path)

            # Write to SQLite
            # if_exists='replace' handles the "TRUNCATE" part automatically
            table_name = f"{folder.split('_')[1]}_{file_name}"
            df.to_sql(table_name, conn, if_exists='replace', index=False)

            duration = round(time.time() - start_time, 2)
            print(f">> Loaded {table_name} in {duration} seconds")
        else:
            print(f"!! File not found: {file_path}")

conn.close()
print(f"--- Load Completed in {round(time.time() - total_start, 2)}s ---")


--- Starting Load to SQLite ---
>> Loaded crm_cust_info in 0.43 seconds
>> Loaded crm_prd_info in 0.08 seconds
>> Loaded crm_sales_details in 0.51 seconds
>> Loaded erp_loc_a101 in 0.18 seconds
>> Loaded erp_cust_az12 in 0.12 seconds
>> Loaded erp_px_cat_g1v2 in 0.04 seconds
--- Load Completed in 1.39s ---


# Silver data

## Tables creation DDL

In [42]:
# Check for unwanted spaces

import pandas as pd
import sqlite3

conn = sqlite3.connect("Datawarehouse.db")

query = """
SELECT *
FROM erp_cust_az12
"""

df = pd.read_sql(query, conn)

conn.close()

df

,CID,BDATE,GEN
0,NASAW00011000,1971-10-06,Male
1,NASAW00011001,1976-05-10,Male
2,NASAW00011002,1971-02-09,Male
3,NASAW00011003,1973-08-14,Female
4,NASAW00011004,1979-08-05,Female
...,...,...,...
18479,AW00029479,1969-06-30,None
18480,AW00029480,1977-05-06,None
18481,AW00029481,1965-07-04,None
18482,AW00029482,1964-09-01,None


## Sivler load data

In [ ]:
conn = sqlite3.connect("Datawarehouse.db")

create_table_query = """
CREATE TABLE IF NOT EXISTS silver_crm_cust_info (
    cst_id INTEGER,
    cst_key TEXT,
    cst_firstname TEXT,
    cst_lastname TEXT,
    cst_marital_status TEXT,
    cst_gndr TEXT,
    cst_create_date TEXT,
    dwh_create_date TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

insert_query = """
INSERT INTO silver_crm_cust_info (
    cst_id,
    cst_key,
    cst_firstname,
    cst_lastname,
    cst_marital_status,
    cst_gndr,
    cst_create_date
)
SELECT
    cst_id,
    cst_key,
    TRIM(cst_firstname) AS cst_firstname,
    TRIM(cst_lastname) AS cst_lastname,
    CASE
        WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
        WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
        ELSE 'n/a'
    END AS cst_marital_status,
    CASE
        WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
        WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
        ELSE 'n/a'
    END AS cst_gndr,
    cst_create_date
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY cst_id
               ORDER BY cst_create_date DESC
           ) AS flag_last
    FROM crm_cust_info
) t
WHERE flag_last = 1
"""

preview_query = """
SELECT *
FROM silver_crm_cust_info
LIMIT 10
"""

conn.execute(create_table_query)
conn.execute("DELETE FROM silver_crm_cust_info")
conn.execute(insert_query)
conn.commit()

df2 = pd.read_sql(preview_query, conn)
conn.close()

df2

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
0,NaN,SF566,None,None,n/a,n/a,None
1,11000.0,AW00011000,Jon,Yang,Married,Male,2025-10-06
2,11001.0,AW00011001,Eugene,Huang,Single,Male,2025-10-06
3,11002.0,AW00011002,Ruben,Torres,Married,Male,2025-10-06
4,11003.0,AW00011003,Christy,Zhu,Single,Female,2025-10-06
5,11004.0,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
6,11005.0,AW00011005,Julio,Ruiz,Single,Male,2025-10-06
7,11006.0,AW00011006,Janet,Alvarez,Single,Female,2025-10-06
8,11007.0,AW00011007,Marco,Mehta,Married,Male,2025-10-06
9,11008.0,AW00011008,Rob,Verhoff,Single,Female,2025-10-06


In [38]:
try:
    conn.close()
except Exception:
    pass

create_table_query = """
CREATE TABLE IF NOT EXISTS silver_crm_prd_info (
    prd_id INTEGER,
    cat_id TEXT,
    prd_key TEXT,
    prd_nm TEXT,
    prd_cost REAL,
    prd_line TEXT,
    prd_start_dt TEXT,
    prd_end_dt TEXT,
    dwh_create_date TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

insert_query = """
INSERT INTO silver_crm_prd_info (
    prd_id,
    cat_id,
    prd_key,
    prd_nm,
    prd_cost,
    prd_line,
    prd_start_dt,
    prd_end_dt
)
SELECT
    prd_id,
    REPLACE(substr(prd_key, 1, 5), '-', '_') AS cat_id,
    substr(prd_key, 7) AS prd_key,
    TRIM(prd_nm) AS prd_nm,
    COALESCE(prd_cost, 0) AS prd_cost,
    CASE
        WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
        WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
        WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'Other Sales'
        WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
        ELSE 'n/a'
    END AS prd_line,
    date(prd_start_dt) AS prd_start_dt,
    COALESCE(
        date(
            LEAD(prd_start_dt) OVER (
                PARTITION BY prd_key
                ORDER BY prd_start_dt
            ),
            '-1 day'
        ),
        date(prd_end_dt)
    ) AS prd_end_dt
FROM crm_prd_info
"""

preview_query = """
SELECT *
FROM silver_crm_prd_info
LIMIT 10
"""

with sqlite3.connect("Datawarehouse.db", timeout=30) as conn:
    conn.execute(create_table_query)
    conn.execute("DELETE FROM silver_crm_prd_info")
    conn.execute(insert_query)
    df3 = pd.read_sql(preview_query, conn)

df3

,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,dwh_create_date
0,478,AC_BC,BC-M005,Mountain Bottle Cage,4.0,Mountain,2013-07-01,None,2026-05-02 05:03:37
1,479,AC_BC,BC-R205,Road Bottle Cage,3.0,Road,2013-07-01,None,2026-05-02 05:03:37
2,477,AC_BC,WB-H098,Water Bottle - 30 oz.,2.0,Other Sales,2013-07-01,None,2026-05-02 05:03:37
3,483,AC_BR,RA-H123,Hitch Rack - 4-Bike,45.0,Other Sales,2013-07-01,None,2026-05-02 05:03:37
4,486,AC_BS,ST-1401,All-Purpose Bike Stand,59.0,Mountain,2013-07-01,None,2026-05-02 05:03:37
5,484,AC_CL,CL-9009,Bike Wash - Dissolver,3.0,Other Sales,2013-07-01,None,2026-05-02 05:03:37
6,485,AC_FE,FE-6654,Fender Set - Mountain,8.0,Mountain,2013-07-01,None,2026-05-02 05:03:37
7,215,AC_HE,HL-U509,Sport-100 Helmet- Black,12.0,Other Sales,2011-07-01,2012-06-30,2026-05-02 05:03:37
8,216,AC_HE,HL-U509,Sport-100 Helmet- Black,14.0,Other Sales,2012-07-01,2013-06-30,2026-05-02 05:03:37
9,217,AC_HE,HL-U509,Sport-100 Helmet- Black,13.0,Other Sales,2013-07-01,None,2026-05-02 05:03:37


In [40]:
try:
    conn.close()
except Exception:
    pass

create_table_query = """
CREATE TABLE IF NOT EXISTS silver_crm_sales_details (
    sls_ord_num TEXT,
    sls_prd_key TEXT,
    sls_cust_id INTEGER,
    sls_order_dt TEXT,
    sls_ship_dt TEXT,
    sls_due_dt TEXT,
    sls_sales REAL,
    sls_quantity INTEGER,
      sls_price REAL,
    dwh_create_date TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

insert_query = """
INSERT INTO silver_crm_sales_details (
    sls_ord_num,
    sls_prd_key,
    sls_cust_id,
    sls_order_dt,
    sls_ship_dt,
    sls_due_dt,
    sls_sales,
    sls_quantity,
    sls_price
)
SELECT
    sls_ord_num,
    sls_prd_key,
    sls_cust_id,
    CASE
        WHEN sls_order_dt = 0 OR length(CAST(sls_order_dt AS TEXT)) != 8 THEN NULL
        ELSE date(substr(CAST(sls_order_dt AS TEXT), 1, 4) || '-' || substr(CAST(sls_order_dt AS TEXT), 5, 2) || '-' || substr(CAST(sls_order_dt AS TEXT), 7, 2))
    END AS sls_order_dt,
    CASE
        WHEN sls_ship_dt = 0 OR length(CAST(sls_ship_dt AS TEXT)) != 8 THEN NULL
        ELSE date(substr(CAST(sls_ship_dt AS TEXT), 1, 4) || '-' || substr(CAST(sls_ship_dt AS TEXT), 5, 2) || '-' || substr(CAST(sls_ship_dt AS TEXT), 7, 2))
    END AS sls_ship_dt,
    CASE
        WHEN sls_due_dt = 0 OR length(CAST(sls_due_dt AS TEXT)) != 8 THEN NULL
        ELSE date(substr(CAST(sls_due_dt AS TEXT), 1, 4) || '-' || substr(CAST(sls_due_dt AS TEXT), 5, 2) || '-' || substr(CAST(sls_due_dt AS TEXT), 7, 2))
    END AS sls_due_dt,
    CASE
        WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
            THEN sls_quantity * ABS(sls_price)
        ELSE sls_sales
    END AS sls_sales,
    sls_quantity,
    CASE
        WHEN sls_price IS NULL OR sls_price <= 0
            THEN sls_sales / NULLIF(sls_quantity, 0)
        ELSE sls_price
    END AS sls_price
FROM crm_sales_details
"""

preview_query = """
SELECT *
FROM silver_crm_sales_details
LIMIT 10
"""

with sqlite3.connect("Datawarehouse.db", timeout=30) as conn:
    conn.execute(create_table_query)
    conn.execute("DELETE FROM silver_crm_sales_details")
    conn.execute(insert_query)
    df3 = pd.read_sql(preview_query, conn)

df3

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price,dwh_create_date
0,SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578.0,1,3578.0,2026-05-02 05:10:34
1,SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,2026-05-02 05:10:34
2,SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,2026-05-02 05:10:34
3,SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0,2026-05-02 05:10:34
4,SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,2026-05-02 05:10:34
5,SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0,2026-05-02 05:10:34
6,SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0,2026-05-02 05:10:34
7,SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375.0,1,3375.0,2026-05-02 05:10:34
8,SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400.0,1,3400.0,2026-05-02 05:10:34
9,SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578.0,1,3578.0,2026-05-02 05:10:34


### erp

In [49]:
try:
    conn.close()
except Exception:
    pass

create_table_query = """
CREATE TABLE IF NOT EXISTS silver_erp_cust_az12 (
    cid TEXT,
    bdate TEXT,
    gen TEXT,
    dwh_create_date TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

insert_query = """
INSERT INTO silver_erp_cust_az12 (
    cid,
    bdate,
    gen
)
SELECT
    CASE
        WHEN cid LIKE 'NAS%' THEN substr(cid, 4)
        ELSE cid
    END AS cid,
    CASE
        WHEN date(bdate) > date('now') THEN NULL
        ELSE date(bdate)
    END AS bdate,
    CASE
        WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
        WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
        ELSE 'n/a'
    END AS gen
FROM erp_cust_az12
"""

preview_query = """
SELECT *
FROM silver_erp_cust_az12
LIMIT 10
"""

with sqlite3.connect("Datawarehouse.db", timeout=30) as conn:
    conn.execute("PRAGMA busy_timeout = 30000")
    conn.execute(create_table_query)
    conn.execute("DELETE FROM silver_erp_cust_az12")
    conn.execute(insert_query)
    df2 = pd.read_sql(preview_query, conn)

df2

,cid,bdate,gen,dwh_create_date
0,AW00011000,1971-10-06,Male,2026-05-02 05:19:46
1,AW00011001,1976-05-10,Male,2026-05-02 05:19:46
2,AW00011002,1971-02-09,Male,2026-05-02 05:19:46
3,AW00011003,1973-08-14,Female,2026-05-02 05:19:46
4,AW00011004,1979-08-05,Female,2026-05-02 05:19:46
5,AW00011005,1976-08-01,Male,2026-05-02 05:19:46
6,AW00011006,1976-12-02,Female,2026-05-02 05:19:46
7,AW00011007,1969-11-06,Male,2026-05-02 05:19:46
8,AW00011008,1975-07-04,Female,2026-05-02 05:19:46
9,AW00011009,1969-09-29,Male,2026-05-02 05:19:46


In [50]:
try:
    conn.close()
except Exception:
    pass

create_table_query = """
CREATE TABLE IF NOT EXISTS silver_erp_loc_a101 (
    cid TEXT,
    cntry TEXT,
    dwh_create_date TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

insert_query = """
INSERT INTO silver_erp_loc_a101 (
			cid,
			cntry
		)
		SELECT
			REPLACE(cid, '-', '') AS cid, 
			CASE
				WHEN TRIM(cntry) = 'DE' THEN 'Germany'
				WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
				WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'n/a'
				ELSE TRIM(cntry)
			END AS cntry -- Normalize and Handle missing or blank country codes
		FROM erp_loc_a101;

"""

preview_query = """
SELECT *
FROM silver_erp_loc_a101
LIMIT 10
"""

with sqlite3.connect("Datawarehouse.db", timeout=30) as conn:
    conn.execute("PRAGMA busy_timeout = 30000")
    conn.execute(create_table_query)
    conn.execute("DELETE FROM silver_erp_loc_a101")
    conn.execute(insert_query)
    df2 = pd.read_sql(preview_query, conn)

df2

,cid,cntry,dwh_create_date
0,AW00011000,Australia,2026-05-02 05:30:52
1,AW00011001,Australia,2026-05-02 05:30:52
2,AW00011002,Australia,2026-05-02 05:30:52
3,AW00011003,Australia,2026-05-02 05:30:52
4,AW00011004,Australia,2026-05-02 05:30:52
5,AW00011005,Australia,2026-05-02 05:30:52
6,AW00011006,Australia,2026-05-02 05:30:52
7,AW00011007,Australia,2026-05-02 05:30:52
8,AW00011008,Australia,2026-05-02 05:30:52
9,AW00011009,Australia,2026-05-02 05:30:52


In [52]:
try:
    conn.close()
except Exception:
    pass

create_table_query = """
CREATE TABLE IF NOT EXISTS silver_erp_px_cat_g1v2 (
    id INTEGER,
    cat TEXT,
    subcat TEXT,
    maintenance TEXT,
    dwh_create_date TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

insert_query = """
INSERT INTO silver_erp_px_cat_g1v2 (
			id,
			cat,
			subcat,
			maintenance
		)
		SELECT
			id,
			cat,
			subcat,
			maintenance
		FROM erp_px_cat_g1v2;
"""

preview_query = """
SELECT *
FROM silver_erp_px_cat_g1v2
LIMIT 10
"""

with sqlite3.connect("Datawarehouse.db", timeout=30) as conn:
    conn.execute("PRAGMA busy_timeout = 30000")
    conn.execute(create_table_query)
    conn.execute("DELETE FROM silver_erp_px_cat_g1v2")
    conn.execute(insert_query)
    df2 = pd.read_sql(preview_query, conn)

df2

,id,cat,subcat,maintenance,dwh_create_date
0,AC_BR,Accessories,Bike Racks,Yes,2026-05-02 05:32:25
1,AC_BS,Accessories,Bike Stands,No,2026-05-02 05:32:25
2,AC_BC,Accessories,Bottles and Cages,No,2026-05-02 05:32:25
3,AC_CL,Accessories,Cleaners,Yes,2026-05-02 05:32:25
4,AC_FE,Accessories,Fenders,No,2026-05-02 05:32:25
5,AC_HE,Accessories,Helmets,Yes,2026-05-02 05:32:25
6,AC_HP,Accessories,Hydration Packs,No,2026-05-02 05:32:25
7,AC_LI,Accessories,Lights,Yes,2026-05-02 05:32:25
8,AC_LO,Accessories,Locks,Yes,2026-05-02 05:32:25
9,AC_PA,Accessories,Panniers,No,2026-05-02 05:32:25
